In [ ]:
# Cell 1 — Login to Hugging Face
# Add your token: 🔑 Secrets (left sidebar) → Name: HF_TOKEN → toggle notebook access ON
from google.colab import userdata
from huggingface_hub import login
hf_token = 'replace'
# hf_token = userdata.get("HF_TOKEN")
login(hf_token)
print("✓ Logged in to Hugging Face")

✓ Logged in to Hugging Face


In [8]:
# Cell 2 — Install (NO optimum — pure torch + onnxruntime only)
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q onnx onnxruntime-gpu accelerate huggingface_hub
!pip install -q onnxscript  # required by torch.onnx dynamo exporter for complex models
print("✅ Done — restart kernel now, then re-run from Cell 1")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 12.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 23.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
optimum-onnx 0.1.0.dev0 requires transformers<4.58.0,>=4.36, but you have transformers 5.6.0.dev0 which is incompatible.
✅ Done — restart kernel now, then re-run from Cell 1


In [2]:
# Cell 3 — Config
MODEL_ID      = "madhured/irs-gemma4-e2b-merged-unsloth"
ONNX_DIR      = "./onnx_model"
ONNX_HUB_REPO = "madhured/irs-gemma4-e2b-onnx"
print(f"Model  : {MODEL_ID}")
print(f"Output : {ONNX_DIR}")

Model  : madhured/irs-gemma4-e2b-merged-unsloth
Output : ./onnx_model


In [3]:
# Cell 4 — Export to ONNX using dynamo exporter
# dynamo=True uses torch.export (not TorchScript) — handles Gemma 4's complex
# dynamic masking and multimodal control flow that TorchScript cannot trace.
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.makedirs(ONNX_DIR, exist_ok=True)

print("Loading tokenizer + model in fp16 on H100...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True,
)
model.eval()
model.config.use_cache = False

# Save tokenizer files (Transformers.js needs these alongside the ONNX weights)
tokenizer.save_pretrained(ONNX_DIR)

# Text-only wrapper — passes pixel_values=None so the vision tower is skipped
class TextOnlyWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        out = self.m(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=None,
            return_dict=False,
        )
        return out[0]   # logits only

wrapped = TextOnlyWrapper(model).cuda()

# Dummy input
dummy   = tokenizer("Hello, I need help with my taxes", return_tensors="pt").to("cuda")
inp_ids = dummy["input_ids"]
attn    = dummy["attention_mask"]

# Quick sanity check before exporting
with torch.no_grad():
    out = wrapped(inp_ids, attn)
print(f"Sanity check OK — logits shape: {out.shape}")

onnx_path = os.path.join(ONNX_DIR, "model.onnx")
print(f"\nExporting to {onnx_path} with dynamo exporter — please wait...")

# dynamic_shapes tells the exporter which dims are variable at runtime
batch = torch.export.Dim("batch_size", min=1, max=8)
seq   = torch.export.Dim("sequence_length", min=1, max=4096)

with torch.no_grad():
    torch.onnx.export(
        wrapped,
        args=(inp_ids, attn),
        f=onnx_path,
        dynamo=True,                    # use torch.export — handles Gemma 4 masking
        dynamic_shapes={
            "input_ids":      {0: batch, 1: seq},
            "attention_mask": {0: batch, 1: seq},
        },
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
    )

size_gb = os.path.getsize(onnx_path) / 1e9
print(f"✓ model.onnx saved — {size_gb:.2f} GB")

Loading tokenizer + model in fp16 on H100...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Sanity check OK — logits shape: torch.Size([1, 8, 262144])

Exporting to ./onnx_model/model.onnx with dynamo exporter — please wait...


/tmp/ipykernel_16424/2107028638.py:57: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
W0415 10:49:16.354000 16424 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0415 10:49:16.354000 16424 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0415 10:49:16.355000 16424 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1

[torch.onnx] Obtain model graph for `TextOnlyWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TextOnlyWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:460: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:460: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


Applied 579 of general pattern rewrite rules.
✓ model.onnx saved — 0.01 GB


In [6]:
# Cell 5 — Verify exported files (no quantization needed — model is already fp16)
# quantize_dynamic cannot handle fp16 tensors in external data format.
# The dynamo exporter already produced fp16 weights (~9 GB total) which is half of fp32.
# Transformers.js loads this with  dtype: 'fp16'  → maps to model.onnx + model.onnx.data
import os

print("Files ready to push:")
total = 0
for f in sorted(os.listdir(ONNX_DIR)):
    size = os.path.getsize(os.path.join(ONNX_DIR, f))
    total += size
    print(f"  {f:60s}  {size/1e9:.3f} GB")
print(f"\n  Total: {total/1e9:.2f} GB")
print("\n✓ Ready to push to Hugging Face Hub — run Cell 6")

Files ready to push:
  chat_template.jinja                                           0.000 GB
  model.onnx                                                    0.009 GB
  model.onnx.data                                               9.258 GB
  tokenizer.json                                                0.032 GB
  tokenizer_config.json                                         0.000 GB

  Total: 9.30 GB

✓ Ready to push to Hugging Face Hub — run Cell 6


In [7]:
# Cell 6 — Push to Hugging Face Hub
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(ONNX_HUB_REPO, exist_ok=True, private=False, repo_type="model")
api.upload_folder(
    folder_path=ONNX_DIR,
    repo_id=ONNX_HUB_REPO,
    repo_type="model",
    commit_message="Add ONNX fp16 + int8 quantized model for Transformers.js / WebGPU",
)
print(f"✓ Live at: https://huggingface.co/{ONNX_HUB_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...onnx_model/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

  ...nnx_model/model.onnx.data:   3%|2         |  255MB / 9.26GB            

  ...ent/onnx_model/model.onnx:  88%|########8 | 7.75MB / 8.77MB            

✓ Live at: https://huggingface.co/madhured/irs-gemma4-e2b-onnx


In [8]:
# Cell 6 (fix) — Save missing config.json and generation_config.json to ONNX_DIR, then re-push
import os, json
from transformers import AutoConfig, GenerationConfig
from huggingface_hub import HfApi

# Load and save the model config (Transformers.js requires config.json)
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
config.save_pretrained(ONNX_DIR)

# Also save generation config if present
try:
    gen_config = GenerationConfig.from_pretrained(MODEL_ID)
    gen_config.save_pretrained(ONNX_DIR)
except Exception:
    pass  # optional, not all models have it

print("Files in ONNX_DIR after fix:")
for f in sorted(os.listdir(ONNX_DIR)):
    size = os.path.getsize(os.path.join(ONNX_DIR, f))
    print(f"  {f:50s}  {size/1e6:.2f} MB")

# Push everything (config.json + any previously missing files) to Hub
api = HfApi()
api.upload_folder(
    folder_path=ONNX_DIR,
    repo_id=ONNX_HUB_REPO,
    repo_type="model",
    commit_message="Add missing config.json and generation_config.json",
)
print(f"\n✓ Updated: https://huggingface.co/{ONNX_HUB_REPO}")

Files in ONNX_DIR after fix:
  chat_template.jinja                                 0.02 MB
  config.json                                         0.01 MB
  model.onnx                                          8.77 MB
  model.onnx.data                                     9257.75 MB
  tokenizer.json                                      32.17 MB
  tokenizer_config.json                               0.00 MB


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...onnx_model/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

  ...ent/onnx_model/model.onnx: 100%|##########| 8.77MB / 8.77MB            

  ...nnx_model/model.onnx.data:   4%|4         |  384MB / 9.26GB            


✓ Updated: https://huggingface.co/madhured/irs-gemma4-e2b-onnx


# Cell 7 — Use your model in the browser with Transformers.js + WebGPU

Once the model is on the Hub, paste this into your web app:

```javascript
// npm install @huggingface/transformers

import { pipeline } from '@huggingface/transformers';

const MODEL_REPO = "madhured/irs-gemma4-e2b-onnx";

async function initEngine() {
    try {
        // dtype:'fp16' loads model.onnx + model.onnx.data (the external weights file)
        return await pipeline('text-generation', MODEL_REPO, {
            device: 'webgpu',
            dtype: 'fp16',
            progress_callback: (p) =>
                console.log(`Loading: ${p.progress?.toFixed(1) ?? 0}%`),
        });
    } catch (e) {
        console.warn("WebGPU failed, falling back to WASM (CPU):", e);
        return await pipeline('text-generation', MODEL_REPO, {
            device: 'wasm',
            dtype: 'fp32',
        });
    }
}

async function ask(question) {
    const generator = await initEngine();
    const result = await generator(question, { max_new_tokens: 256 });
    return result[0].generated_text;
}

ask("What documents do I need for IRS tax filing?").then(console.log);
```

### dtype → file mapping
| `dtype` | Files loaded |
|---|---|
| `'fp16'` | `model.onnx` + `model.onnx.data` ← **your model** |
| `'fp32'` | `model.onnx` + `model.onnx.data` (fp32 version) |
| `'q8'`   | `model_quantized.onnx` (int8, smaller but needs separate export) |

> **Note:** A ~9 GB model will take 1–3 minutes to download into the browser's IndexedDB cache on first load. Subsequent loads are instant from cache.
> **WebGPU requires:** Chrome 113+ or Edge 113+ with a dedicated GPU.

Loading model on H100 for inference...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

✓ Model loaded


ERROR:pyngrok.process.ngrok:t=2026-04-15T11:12:22+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T11:12:22+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T11:12:22+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.